In [1]:
import pandas as pd
import numpy as np
from RuleTree import RuleTreeClassifier
from HybridReaders import read_wdbc
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier as knn
from sklearn.linear_model import LogisticRegression as lr
from sklearn.metrics import accuracy_score, classification_report, f1_score, jaccard_score, recall_score, precision_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import kendalltau, spearmanr, pearsonr
import random as rd
from joblib import dump, load
import shap
import os

C:\Users\franc\anaconda3\envs\py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataframe splitting
It splits the df and optionally saves splits. The seed is fixed at 42 for reproducibility on the same df.   
It also optionally scales X

save -> True if we want to save the splits as csv   
name_file -> to change the name of the file depending on the df we are using as input

In [4]:
def split_ts(df_name, df, save = True, scale = True, test_size = 0.3, save_path = './'):
    
    y = df['y'].values
    X = df[df.columns[:-1]].values
    if scale:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    X_tr, X_ts, y_tr, y_ts = train_test_split(X, y, test_size = test_size, random_state = 42, stratify = y)
    to_save = {f'{df_name}_X_tr': X_tr, f'{df_name}_X_ts': X_ts, f'{df_name}_y_tr': y_tr, f'{df_name}_y_ts': y_ts}
    
    if save:
        for key, el in to_save.items():
            try:
                pd.DataFrame(el).to_csv(f'{save_path}/{key}.csv', index=False)
            except FileNotFoundError:
                print('The directory does not exist or it is wrong')
        return X_tr, X_ts, y_tr, y_ts
    else:
        return X_tr, X_ts, y_tr, y_ts

if __name__ == '__main__':
    df_name, df = read_wdbc(basepath = "C:/Users/franc/OneDrive/Desktop/Magistrale/Tesi modelli equivalenti/")
    X_tr, X_ts, y_tr, y_ts = split_ts(df_name = df_name, df = df, save = False, save_path = 'C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Split_salvati')

### Model training
It trains n. models, saves them on a joblib file and prints accuracy and varying parameters just to give a first look

model_type -> the model we want to train (rtc, knn, logreg)
max_models -> n. of models we want to train   
min_d -> min depth for rtc   
max_d -> max depth for rtc

max_nb/ min_nb -> max/min neighbours



In [39]:
def train_models(model_type, df_name, X_tr, X_ts, y_tr, y_ts, max_models = 5, min_d = 3, max_d = 20, save_path = './', max_nb = 5, min_nb = 1):

    np.random.seed(0)
    n_neighbors = np.random.choice(np.arange(min_nb, max_nb +1), size = max_models, replace = False)
    C = np.random.choice(np.logspace(-4, 4), size = max_models, replace = False)
    for i in range(max_models):
        rd.seed(i)
        if model_type == 'rtc':
            model = RuleTreeClassifier(
                max_depth = rd.randint(min_d, max_d),
                criterion = rd.choice(('gini', 'entropy')),
                prune_useless_leaves=True
            )
            params = ['max_depth', 'criterion']
            
        elif model_type == 'knn':
            model = knn(
                n_neighbors = n_neighbors[i],
                weights = rd.choice(('uniform','distance'))
            )
            params = ['n_neighbors', 'weights']

        elif model_type == 'lr':
            model = lr(
                C = C[i],
                max_iter = 500
            )
            params = ['C']

        model.fit(X_tr, y_tr)
        try:
            dump(model, f'{save_path}/{df_name}_{model_type}_{i+1}.joblib')
        
        except FileNotFoundError:
            print('The directory does not exist or it is wrong')
 
        #solo per dare una prima occhiata veloce
        y_pred = model.predict(X_ts)
        print(f'{df_name}_{model_type}_{i+1} \naccuracy: {accuracy_score(y_ts, y_pred)}') 
        for par in params:
            print(f'{par}: {model.get_params()[par]}')
            
if __name__ == '__main__':
    df_name, df = read_wdbc(basepath = "C:/Users/franc/OneDrive/Desktop/Magistrale/Tesi modelli equivalenti/")
    X_tr, X_ts, y_tr, y_ts = split_ts(df_name = df_name, df = df, save = False, save_path = 'C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Split_salvati')
    train_models(model_type = 'knn', df_name = 'wdbc', X_tr = X_tr, X_ts = X_ts, y_tr = y_tr, y_ts = y_ts, save_path = 'C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc')
    
    

wdbc_knn_1 
accuracy: 0.9590643274853801
n_neighbors: 3
weights: distance
wdbc_knn_2 
accuracy: 0.9415204678362573
n_neighbors: 1
weights: uniform
wdbc_knn_3 
accuracy: 0.935672514619883
n_neighbors: 2
weights: uniform
wdbc_knn_4 
accuracy: 0.9590643274853801
n_neighbors: 4
weights: uniform
wdbc_knn_5 
accuracy: 0.9649122807017544
n_neighbors: 5
weights: uniform


### Measures and Functions
le dividiamo in inter/intra family   

**preds_concordance()**   (inter-family)   
prende due modelli, usa la jaccard per definire la concordanza con la ground truth e la concordanza tra loro. in base alla misura scelta dice se c'è concordanza tra i modelli o no. Ci rende un dizionario con le info 

measures: jac_diff, concordance   
parameters:   
measure -> per scegliere quale misura delle due usare   
diff_th -> threshold nel caso di jac_diff   
conc_th -> threshold per la concordance   
average -> uso la jaccard_score di sklearn. Average mi serve per calcolare la jaccard in base al tipo di classificazione



In [56]:
#A PRESCINDERE DALLA FAMIGLIA
#quanto le predizioni sono concordi a ground truth? uno è più concorde dell'altro?
#quanto sono concordi due modelli m1 e m2?
def preds_concordance(m1, m2, X_ts, y_ts, average = 'binary', measure = 'jac_diff', diff_th = 0.2, conc_th = 0.7):

    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

    truth_d1 = jaccard_score(y_ts, y1_pred, average = average) #quanto m1 dista dalla ground truth?
    truth_d2 = jaccard_score(y_ts, y2_pred, average = average) #quanto m2 dista dalla ground truth?
    
    #due misure diverse per la concordanza
    jac_diff = abs(y1_pred - y2_pred) #diff di concordanza con la gt
    concordance = jaccard_score(y1_pred, y2_pred, average = average) #quanto concordano tra loro i modelli?
    
    sim_dict = {
        'm1_distance_from_truth': truth_d1, 
        'm2_distance_from_truth': truth_d2, 
    }

    if measure == 'jac_diff':
            sim_dict['jac_diff'] = jac_diff
            sim_dict['concordant'] = jac_diff <= diff_th
    else:
            sim_dict['concordance'] = concordance
            sim_dict['concordant'] = concordance >= conc_th
    
    return sim_dict

 
if __name__ == '__main__':
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_knn_2.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_1.joblib')
    
    d = preds_concordance(m1, m2, X_ts, y_ts)
    for key, el in d.items():
        print(f'{key}: {el}')
    

m1_distance_from_truth: 0.8529411764705882
m2_distance_from_truth: 0.7746478873239436
jac_diff: 0.07829328914664457
concordant: True


**prediction_confidence** (inter-family)   
prende due modelli, ne considera le previsioni che combaciano e studia se coincidono abbastanza per quanto riguarda la % di confidence   

misure:   
coeff di correlazione (per binary) -> spearman, pearson, kendall a scelta    
cueff di correlazione medio tra classi -> stesse metriche a scelta   

parametri:   
metric -> scegliamo tra le 3 indicate   
tolerance -> livello di tolleranza sul valore del coeff di corr per determinare se la sicurezza nella predizioni è simile

In [52]:
#A PRESCINDERE DALLA FAMIGLIA
#due modelli a parità di previsione, quanto variano nella confidence? sono correlati?
#in caso di predizioni opposte, quanto si discostano in confidence? sono entrambi abbastanza incerti o uno è marcatamente più sicuro? 

def prediction_confidence_corr(m1, m2, y_ts, X_ts, metric = 'spearman', concord_preds = True):

    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

     #filtriamo per predizioni concordanti o discordanti
    if concord_preds:
        y1_prob = m1.predict_proba(X_ts)[y1_pred == y2_pred]
        y2_prob = m2.predict_proba(X_ts)[y1_pred == y2_pred]
    else:
        y1_prob = m1.predict_proba(X_ts)[y1_pred != y2_pred]
        y2_prob = m2.predict_proba(X_ts)[y1_pred != y2_pred]

    #in caso di classificazione binaria
    if y1_prob.shape[1] == 2:
        
        if metric == 'spearman':
            confidence_corr, _ = spearmanr(y1_prob[:, 1], y2_prob[:, 1])
        elif metric == 'pearson':
            confidence_corr, _ = pearsonr(y1_prob[:, 1], y2_prob[:, 1])
        elif metric == 'kendall':
            confidence_corr, _ = kendalltau(y1_prob[:, 1], y2_prob[:, 1])
        
        return confidence_corr

    #in caso multiclasse facciamo un dizionario con la correlazione per classe
    else:
        corrs = {}
        for cl in range(y1_prob.shape[1]):
            if metric == 'spearman':
                confidence_corr, _ = spearmanr(y1_prob[:, cl], y2_prob[:, cl])
            elif metric == 'pearson':
                confidence_corr, _ = pearsonr(y1_prob[:, cl], y2_prob[:, cl])
            elif metric == 'kendall':
                confidence_corr, _ = kendalltau(y1_prob[:, cl], y2_prob[:, cl])
            
            corrs[cl] = confidence_corr
        
        return corrs
            
if __name__ == '__main__':
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_knn_3.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_lr_1.joblib')

    confidence_corr = prediction_confidence_corr(m1, m2, y_ts, X_ts, metric = 'kendall')
    print(confidence_corr)

0.5067713929452493


In [59]:
#stessa cosa di prima, solo che al posto della correlazione guardiamo quanto si discostano tra i vettori con mse/mae
#in caso di predizioni opposte/uguali, quanto si discostano in confidence? sono entrambi abbastanza incerti o uno è marcatamente più sicuro? 

def prediction_confidence_error(m1, m2, y_ts, X_ts, metric = 'mae', concord_preds = True):

    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

     #filtriamo per predizioni concordanti o discordanti
    if concord_preds:
        y1_prob = m1.predict_proba(X_ts)[y1_pred == y2_pred]
        y2_prob = m2.predict_proba(X_ts)[y1_pred == y2_pred]
    else:
        y1_prob = m1.predict_proba(X_ts)[y1_pred != y2_pred]
        y2_prob = m2.predict_proba(X_ts)[y1_pred != y2_pred]

    #in caso di classificazione binaria
    if y1_prob.shape[1] == 2:

        if metric == 'mae':
            error = mean_absolute_error(y1_prob, y2_prob)
        elif metric == 'mse':
            error = mean_squared_error(y1_prob, y2_prob)

        return error
            
    #in caso multiclasse facciamo un dizionario con la correlazione per classe
    else:
        errors = {}
        for cl in range(y1_prob.shape[1]):
            
            if metric == 'mae':
                error = mean_absolute_error(y1_prob, y2_prob)
            elif metric == 'mse':
                error = mean_squared_error(y1_prob, y2_prob)
            
            errors[cl] = error
        
        return errors
            
if __name__ == '__main__':
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_knn_3.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_lr_1.joblib')

    confidence_similarity = prediction_confidence_error(m1, m2, y_ts, X_ts, metric = 'mse')
    print(confidence_similarity)

0.040312587132452185


In [ ]:
#filtrare: quando sono sicuri, quanto sbagliano?

#A PRESCINDERE DALLA FAMIGLIA
#considerando una certa proporzione di rappresentazione per ogni classe, quanto viene considerata ogni classe?

def class_representation(m1, m2, X_ts, y_ts, tolerance = 0.09):

    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

    #quanti esempi di ogni classe sono stati catturati dai modelli
    rec1 = recall_score(y_ts, y1_pred, average = None)
    rec2 = recall_score(y_ts, y2_pred, average = None)

    repr_diff = rec1 - rec2 #differenza nelle recall per classe
    classes = sorted(list(set(y1_pred)))

    repr_dict = dict(zip(classes, repr_diff))
    mean_repr_diff = np.mean(list(repr_dict.values())) #differenza media di tutte le classi

    info_d = {
        'repr_differences': repr_dict,
        'mean_repr_diff': mean_repr_diff,
        'repr_similarity': mean_repr_diff <= tolerance
    }
    return info_d

if __name__ == '__main__':
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_knn_5.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_1.joblib')

    info_d = class_representation(m1, m2, X_ts, y_ts)
    for key, val in info_d.items():
        print(f'{key}: {val}')
        

**feature_influence()** (inter-family)   

misure:   
influence_corr -> calcola il coeff di correlazione spearman tra gli shap vals dei modelli per ogni instance. Se il problema è multiclasse, questo viene calcolato classe per classe. Usiamo spearman perché calcola una correlazione sul rank degli shap vals

parametri:   
concord_preds -> true se si vuole lavorare sulle predizioni concordi, false se si lavora sulle discordi   
m1_fam/m2_fam -> specifichiamo che tipi di modelli sono: (tree, log_reg, knn)

appunti:   
Perché spearman e non pearson per la correlazione?   
perché spearman calcola la correlazione sul rank di valori e non sui valori assoluti degli shap vals. Questo ha più senso se abbiamo davanti due modelli di famiglie diverse, che quindi calcolano gli shap sfruttando proprietà della famiglia del modello, portando quindi a valori in scala diversi (ex. uno shap val di 0.3 per un decision tree non è lo stesso di uno 0.3 per un knn o log.reg, mentre il rank non è relativo da quel punto di vista). Questo discorso non vale se usiamo un explainer model agnostic come kernelExplainer, ma a quel punto possiamo tenere anche spearman senza fare troppi giri

In [ ]:
#quando due modelli concordano, quale feature li influenza di più? C'è un perché diverso?

In [3]:
#A PRESCINDERE DALLA FAMIGLIA
#quando concordano (non concordano), c'è una correlazione nell'influenza delle features? (shap vals) 

def feature_influence(X_ts, m1, m2, m1_fam = 'tree', m2_fam = 'tree', concord_preds = True):
    
    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

    #dipende se vogliamo le predizioni concordanti o discordanti (conc_preds)
    if concord_preds:
        idx = np.where(y1_pred == y2_pred)[0]
    else:
        idx = np.where(y1_pred != y2_pred)[0]

    #da ora usiamo i dati filtrati
    X_ts_filtered = X_ts[idx]
    y1_pred_filtered = y1_pred[idx]
    y2_pred_filtered = y2_pred[idx]

    #un warning di shap l'ha consigliato per velocizzare i calcoli dei valori attesi sulle previsioni dei modelli per il calcolo degli shap vals
    #il kernelExplainer simula l'assenza delle features, quindi ne ha bisogno
    background = shap.kmeans(X_ts_filtered, 20)
    #ci basiamo sulla famiglia del modello per definire un explainer più specifico se possibile
    if m1_fam == 'tree' and m2_fam == 'tree':
        expl1 = shap.TreeExplainer(m1.predict)
        expl2 = shap.TreeExplainer(m2.predict)
    elif m1_fam == 'log_reg' and m2_fam == 'log_reg':
        expl1 = shap.LinearExplainer(m1.predict)
        expl2 = shap.LinearExplainer(m2.predict)
    #il knn non ha un explainer specifico
    elif m1_fam == 'knn' and m2_fam == 'knn':
        expl1 = shap.KernelExplainer(m1.predict, background)
        expl2 = shap.KernelExplainer(m2.predict, background)

    #se i modelli sono di famiglie diverse usiamo il kernelExplainer che è model agnostic
    else:
        expl1 = shap.KernelExplainer(m1.predict, background)
        expl2 = shap.KernelExplainer(m2.predict, background)

    #è più semplice calcolare gli shap vals per intero e poi iterare riga per riga sui valori già calcolati
    shap1_vals = expl1(X_ts_filtered).values #faccio direttamente .values perché mi interessano solo quelli
    shap2_vals = expl2(X_ts_filtered).values

    influence_corr_d = {} 
    n_classes = len(np.unique(y1_pred))
    #la matrice shap values ha queste dimensioni: (instances, features, classes) per multiclasse, (instances, features) per binaria
    #la correlazione calcolata è quella sul rank degli shap vals dato dai modelli per instance
    for i in range(shap1_vals.shape[0]):

        #se il problema è multiclasse, il dizionario di correlazioni sarà {instance: lista correlazioni per classe}
        if n_classes > 2:

            correlations = []
            #la matrice shap values ha queste dimensioni: (instances, features, classes). Mi serve calcolare la corr classe per classe
            for cl in shap1_vals[i].T: #trasposta perché vogliamo iterare per elemento e le classi sono le colonne
                influence_corr, _ = spearmanr(shap1_vals[cl], shap2_vals[cl])
                correlations.append(influence_corr)
            influence_corr_d[i] = correlations
    
        #se il problema è multiclasse, il dizionario di correlazioni sarà {instance: correlazione}
        else:
            influence_corr, _ = spearmanr(shap1_vals[i], shap2_vals[i])
            influence_corr_d[i] = influence_corr
            

    
    return influence_corr_d

        
        



In [4]:
if __name__ == '__main__':
    X_ts = np.genfromtxt('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Split_salvati/vehicle_X_ts.csv', delimiter=',', skip_header=1)
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_3.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_1.joblib')
    influence_corr_dict = feature_influence(X_ts, m1, m2, m1_fam = 'rtc', m2_fam = 'rtc', concord_preds = True)
    print(influence_corr_dict)

IndexError: index 22 is out of bounds for axis 1 with size 18

In [7]:
#questa è la stessa di prima ma con gli shap vals calcolati sui centroidi per alleggerire il dizionario
#inseriamo un parametro n_instances per controllare il numero di centroidi che vogliamo
def feature_influence_sampled(X_ts, m1, m2, m1_fam = 'tree', m2_fam = 'tree', concord_preds = True, n_instances = 20):
    
    y1_pred = m1.predict(X_ts)
    y2_pred = m2.predict(X_ts)

    if concord_preds:
        idx = np.where(y1_pred == y2_pred)[0]
    else:
        idx = np.where(y1_pred != y2_pred)[0]

    X_ts_filtered = X_ts[idx]
    y1_pred_filtered = y1_pred[idx]
    y2_pred_filtered = y2_pred[idx]

    background = shap.sample(X_ts_filtered, n_instances) #con n_instances controlliamo il numero di centroidi che vogliamo
    if m1_fam == 'tree' and m2_fam == 'tree':
        expl1 = shap.TreeExplainer(m1.predict) 
        expl2 = shap.TreeExplainer(m2.predict)
    elif m1_fam == 'log_reg' and m2_fam == 'log_reg':
        expl1 = shap.LinearExplainer(m1.predict)
        expl2 = shap.LinearExplainer(m2.predict)
    elif m1_fam == 'knn' and m2_fam == 'knn':
        expl1 = shap.KernelExplainer(m1.predict, background)
        expl2 = shap.KernelExplainer(m2.predict, background)
    else:
        expl1 = shap.KernelExplainer(m1.predict, background)
        expl2 = shap.KernelExplainer(m2.predict, background)

    #calcolo gli shap vals sui centroidi
    shap1_vals = expl1(background).values 
    shap2_vals = expl2(background).values

    influence_corr_d = {} 
    correlations = []
    n_classes = len(np.unique(y1_pred))

    for i in range(shap1_vals.shape[0]):
        
        if n_classes > 2:
            for cl in shap1_vals.T: 
                influence_corr, _ = spearmanr(shap1_vals[cl], shap2_vals[cl])
                correlations.append(influence_corr)
            influence_corr_d[i] = correlations
        else:
            influence_corr, _ = spearmanr(shap1_vals[i], shap2_vals[i])
            influence_corr_d[i] = influence_corr
            
    return influence_corr_d

        
        
if __name__ == '__main__':
    X_ts = np.genfromtxt('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Split_salvati/wdbc_X_ts.csv', delimiter=',', skip_header=1)
    m1 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_3.joblib')
    m2 = load('C:/Users/franc/OneDrive/Desktop/Magistrale/Thesis-Model-Equivalence/Models/wdbc/wdbc_rtc_1.joblib')
    influence_corr_dict = feature_influence(X_ts, m1, m2, m1_fam = 'tree', m2_fam = 'tree', concord_preds = True)
    print(influence_corr_dict)


    

InvalidModelError: Model type not yet supported by TreeExplainer: <class 'method'>